# Notebook 04: Calcification Detection — YOLO, Segmentation and Radiomics

This notebook covers the second and more specialized diagnostic pipeline: detecting calcium
deposits in the rotator cuff. The pipeline chains four stages — anatomical detection, calcium
segmentation, radiomics feature extraction, and morphological classification.

**What you will learn:**
- Why two separate YOLO models are used (internal vs external rotation views)
- How YOLO performs object detection and how confidence scores drive the decision logic
- How a dedicated U-Net segments calcium deposits (different from the shoulder crop U-Net)
- How 23 hand-crafted radiomics features (morphological, intensity, texture) characterize a lesion
- How a logistic regression classifier maps those features to a Gartner type prediction

**Source code references:**
- `Backend/app.py` — `/predict_yolo` endpoint
- `Backend/calcium_unet.py` — `CalciumUNet`
- `Backend/calcium_radiomics.py` — `CalciumRadiomics`
- `Backend/calcium_classifier.py` — `CalciumClassifier`
- `Backend/Calcium_Pipeline.py` — `CalciumPipeline`

> **Note on model weights:** YOLO and calcium models are downloaded at runtime. Inference cells
> will skip gracefully if no weights are present. The radiomics section is fully runnable without
> any trained model.

## 1. Clinical Context: Rotator Cuff Anatomy and Projections

The rotator cuff consists of four tendons. The two most commonly affected by calcifications are:

- **Supraspinatus (supr):** runs above the humeral head and inserts at the greater tuberosity.
  It is the most frequently calcified tendon (~80% of cases) and is best visualized in the
  external rotation (RE) AP projection.

- **Infraspinatus (infr):** runs posterior to the humeral head and also inserts at the greater
  tuberosity but slightly inferior. It is better visualized in the internal rotation (RI)
  AP projection.

### Why two YOLO models?
The appearance of the tendon insertion area changes significantly between RI and RE projections
due to the rotation of the humeral head. Training a single YOLO model on both projections mixed
together would force it to learn two different visual representations of the same anatomy.

Instead, two specialized models are trained:
- `yolo_re` — detects landmarks in external rotation images
- `yolo_ri` — detects landmarks in internal rotation images

Both models run in parallel on each input image. The detection with the highest confidence score
above a safety threshold is selected as the final output. This way, the system works regardless
of which projection was taken, without requiring the user to specify it.

## 2. Imports and Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import os
from skimage.measure import regionprops, label
from skimage.feature import graycomatrix, graycoprops

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError:
    YOLO_AVAILABLE = False
    print("ultralytics not installed. YOLO inference cells will be skipped.")

try:
    import torch
    import segmentation_models_pytorch as smp
    TORCH_AVAILABLE = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available. Calcium U-Net inference will be skipped.")

JPG_PATH         = "../Frontend/Images/shoulder_x-ray.jpg"
YOLO_RE_WEIGHTS  = None   # Path to best_RI.pt if available
YOLO_RI_WEIGHTS  = None   # Path to best.pt if available
CALCIUM_UNET_PTH = None   # Path to unet_pesos_windows.pth if available
CALCIUM_LR_PATH  = None   # Path to clf_lr.joblib if available

In [ ]:
img_raw = cv2.imread(JPG_PATH, cv2.IMREAD_GRAYSCALE)
assert img_raw is not None
img_norm = img_raw.astype(np.float32)
img_uint8 = ((img_norm - img_norm.min()) / (img_norm.max() - img_norm.min()) * 255).astype(np.uint8)

# YOLO expects RGB at 896x896
img_896 = cv2.resize(img_uint8, (896, 896), interpolation=cv2.INTER_AREA)
img_rgb = cv2.cvtColor(img_896, cv2.COLOR_GRAY2RGB)

print(f"YOLO input shape: {img_rgb.shape}")

## 3. YOLO Anatomical Detection

YOLO (You Only Look Once) is a single-stage object detector that predicts bounding boxes and
class probabilities in a single forward pass. Unlike two-stage detectors (e.g., Faster R-CNN),
YOLO applies the network once to the full image and regresses box coordinates directly from
the feature map grid, making it much faster.

In this project, both YOLO models are fine-tuned from YOLOv8 pretrained weights to detect
two anatomical classes:

- `supr` — supraspinatus tendon insertion region
- `infr` — infraspinatus tendon insertion region

### Detection thresholds
Two confidence thresholds are used:
- `conf_min_detect = 0.15` — minimum confidence for YOLO to even produce a box output
- `SAFE_THRESHOLD = 0.40` — minimum confidence for a detection to be accepted as final output

Only detections above the safe threshold trigger the downstream calcium pipeline. Detections
between the two thresholds are discarded. This two-level approach reduces false positives
while still allowing the NMS (non-maximum suppression) step to consider more candidates.

In [ ]:
yolo_re = None
yolo_ri = None

if YOLO_AVAILABLE:
    if YOLO_RE_WEIGHTS and os.path.exists(YOLO_RE_WEIGHTS):
        yolo_re = YOLO(YOLO_RE_WEIGHTS)
        print(f"YOLO RE loaded: classes = {yolo_re.names}")
    if YOLO_RI_WEIGHTS and os.path.exists(YOLO_RI_WEIGHTS):
        yolo_ri = YOLO(YOLO_RI_WEIGHTS)
        print(f"YOLO RI loaded: classes = {yolo_ri.names}")
else:
    print("YOLO not available.")

In [ ]:
CONF_MIN = 0.15
SAFE_THR = 0.40

best_detection = None

def run_and_select(yolo_model, img_rgb, conf_min, safe_thr, current_best):
    if yolo_model is None:
        return current_best
    results = yolo_model.predict(img_rgb, conf=conf_min, imgsz=896)
    if not results or len(results[0].boxes) == 0:
        return current_best
    for i in range(len(results[0].boxes)):
        conf = float(results[0].boxes.conf[i].item())
        if conf < safe_thr:
            continue
        if current_best is None or conf > current_best["conf"]:
            cls_name = yolo_model.names[int(results[0].boxes.cls[i].item())]
            current_best = {
                "cls_name": cls_name,
                "conf": conf,
                "xyxy": results[0].boxes.xyxy[i].cpu().numpy().astype(int),
            }
    return current_best

if yolo_re is not None or yolo_ri is not None:
    best_detection = run_and_select(yolo_re, img_rgb, CONF_MIN, SAFE_THR, best_detection)
    best_detection = run_and_select(yolo_ri, img_rgb, CONF_MIN, SAFE_THR, best_detection)

    display_img = img_rgb.copy()
    if best_detection:
        x1, y1, x2, y2 = best_detection["xyxy"]
        label_text = f"{best_detection['cls_name']} {best_detection['conf']:.2f}"
        cv2.rectangle(display_img, (x1, y1), (x2, y2), (0, 200, 0), 3)
        cv2.putText(display_img, label_text, (x1, max(y1-10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)
        print(f"Best detection: {best_detection['cls_name']} (conf={best_detection['conf']:.3f})")
    else:
        print("No detection above the safe threshold.")

    plt.figure(figsize=(6, 6))
    plt.imshow(display_img)
    plt.title("YOLO detection result")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping YOLO inference (no model weights).")
    print("Expected output: bounding box around supr or infr region with confidence score.")

## 4. Calcium Deposit Segmentation (U-Net)

When YOLO detects a supraspinatus or infraspinatus region above the safe threshold, the
corresponding image crop is passed to a dedicated **calcium U-Net** — a separate model from
the shoulder cropping U-Net in Notebook 02.

### Differences from the shoulder crop U-Net

| Property | Shoulder crop U-Net | Calcium U-Net |
|----------|--------------------|--------------|
| Input channels | 3 (RGB) | 1 (grayscale) |
| Task | Segment humeral head region | Segment calcium deposit pixels |
| Encoder | EfficientNet-B2 | EfficientNet-B2 |
| Weights format | .ckpt (PyTorch Lightning) | .pth (plain state dict) |
| Output | 512x512 mask (shoulder region) | 512x512 mask (calcium pixels) |

The calcium U-Net uses a single-channel input because grayscale images better represent the
X-ray attenuation values that distinguish calcium from surrounding soft tissue. The model
predicts a probability map; a threshold of 0.5 converts it to a binary mask.

In [ ]:
calcium_unet = None

if TORCH_AVAILABLE and CALCIUM_UNET_PTH and os.path.exists(CALCIUM_UNET_PTH):
    calcium_model = smp.UnetPlusPlus(
        encoder_name="efficientnet-b2",
        encoder_weights=None,
        in_channels=1,    # grayscale
        classes=1,
    )
    state_dict = torch.load(CALCIUM_UNET_PTH, map_location=device)
    calcium_model.load_state_dict(state_dict)
    calcium_model.to(device).eval()
    calcium_unet = calcium_model
    print("Calcium U-Net loaded.")
else:
    print("Calcium U-Net weights not available. Segmentation section will be conceptual.")


def predict_calcium_mask(img_uint8_gray, model, threshold=0.5):
    img_512 = cv2.resize(img_uint8_gray, (512, 512), interpolation=cv2.INTER_AREA)
    img_norm = img_512.astype("float32") / 255.0

    # Shape: (1, 1, 512, 512) — batch x channels x H x W
    tensor = torch.from_numpy(img_norm).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

    mask_bin = (prob > threshold).astype(np.uint8) * 255
    return mask_bin, prob


if calcium_unet is not None:
    mask_bin, prob_map = predict_calcium_mask(img_uint8, calcium_unet)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_uint8, cmap="gray")
    axes[0].set_title("Input image")
    axes[0].axis("off")
    axes[1].imshow(prob_map, cmap="hot", vmin=0, vmax=1)
    axes[1].set_title("Calcium probability map")
    axes[1].axis("off")
    axes[2].imshow(mask_bin, cmap="gray")
    axes[2].set_title("Binary mask (threshold=0.5)")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Creating a synthetic example mask for the radiomics demonstration below.")
    # Synthetic elliptical mask for demonstration
    mask_bin = np.zeros((img_uint8.shape[0], img_uint8.shape[1]), dtype=np.uint8)
    h, w = img_uint8.shape
    cv2.ellipse(mask_bin, (w//2, h//3), (w//12, h//20), 0, 0, 360, 255, -1)
    print(f"Synthetic mask created: {mask_bin.sum() // 255} foreground pixels")

## 5. Radiomics Feature Extraction

Radiomics is the process of extracting large numbers of quantitative features from medical
images. Unlike deep learning, which learns features automatically, radiomics features are
hand-crafted based on domain knowledge about what visually distinguishes different lesion types.

For the Gartner classification of calcifications, the relevant features describe:

### 5.1 Morphological features (5 features)
Derived from the connected region properties of the binary mask:
- `area_px` — total number of foreground pixels
- `eccentricity` — how elongated the region is (0 = circle, 1 = line)
- `solidity` — ratio of region area to convex hull area (measures concavity)
- `perimeter` — length of the region boundary
- `bbox_area` — area of the bounding box enclosing the region

### 5.2 Intensity features (6 features)
Computed on the X-ray pixel values within the foreground mask:
- `int_mean`, `int_std`, `int_min`, `int_max` — basic statistics
- `int_p10`, `int_p90` — 10th and 90th percentiles (robust to outliers)

### 5.3 GLCM texture features (12 features)
The Gray-Level Co-occurrence Matrix (GLCM) encodes the frequency with which pairs of pixels
with specific intensity values appear at a given distance and angle. It captures the spatial
arrangement of intensities — whether the deposit looks homogeneous or heterogeneous.

For each of 6 GLCM properties, mean and standard deviation are computed across 4 directions
(0, 45, 90, 135 degrees) to achieve rotation invariance:
- `contrast` — local intensity variation
- `dissimilarity` — weighted difference of pixel pair intensities
- `homogeneity` — closeness of distribution to diagonal
- `energy` — uniformity (also called Angular Second Moment)
- `correlation` — linear dependency between pixel pairs
- `ASM` — Angular Second Moment (closely related to energy)

**Total: 5 + 6 + 12 = 23 features** — the exact number expected by the logistic regression model.

In [ ]:
def extract_radiomics(image_uint8, mask_uint8):
    mask = (mask_uint8 > 0).astype(np.uint8)
    if mask.sum() == 0:
        raise ValueError("Mask is empty — cannot extract radiomics.")

    # Keep only the largest connected component
    labeled = label(mask)
    largest = np.argmax(np.bincount(labeled.flat)[1:]) + 1
    mask = (labeled == largest).astype(np.uint8)

    props = regionprops(mask)[0]

    # --- Morphological ---
    area_px     = props.area
    eccentricity = props.eccentricity
    solidity    = props.solidity
    perimeter   = props.perimeter
    bbox_area   = props.bbox_area

    # --- Intensity ---
    image_f = image_uint8.astype("float32") / 255.0
    pixels  = image_f[mask == 1]
    int_mean = np.mean(pixels)
    int_std  = np.std(pixels)
    int_min  = np.min(pixels)
    int_max  = np.max(pixels)
    int_p10  = np.percentile(pixels, 10)
    int_p90  = np.percentile(pixels, 90)

    # --- GLCM ---
    img_q = (image_f * 255).astype(np.uint8) * mask
    r0, c0, r1, c1 = props.bbox
    img_crop = img_q[r0:r1, c0:c1]

    glcm = graycomatrix(
        img_crop,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True,
    )

    def glcm_stat(prop):
        v = graycoprops(glcm, prop).flatten()
        return np.mean(v), np.std(v)

    contrast_m,      contrast_s      = glcm_stat("contrast")
    dissimilarity_m, dissimilarity_s = glcm_stat("dissimilarity")
    homogeneity_m,   homogeneity_s   = glcm_stat("homogeneity")
    energy_m,        energy_s        = glcm_stat("energy")
    correlation_m,   correlation_s   = glcm_stat("correlation")
    asm_m,           asm_s           = glcm_stat("ASM")

    return {
        "area_px": area_px, "eccentricity": eccentricity,
        "solidity": solidity, "perimeter": perimeter, "bbox_area": bbox_area,
        "int_mean": int_mean, "int_std": int_std,
        "int_min": int_min,  "int_max": int_max,
        "int_p10": int_p10,  "int_p90": int_p90,
        "glcm_contrast_mean": contrast_m,         "glcm_contrast_std": contrast_s,
        "glcm_dissimilarity_mean": dissimilarity_m, "glcm_dissimilarity_std": dissimilarity_s,
        "glcm_homogeneity_mean": homogeneity_m,   "glcm_homogeneity_std": homogeneity_s,
        "glcm_energy_mean": energy_m,             "glcm_energy_std": energy_s,
        "glcm_correlation_mean": correlation_m,   "glcm_correlation_std": correlation_s,
        "glcm_ASM_mean": asm_m,                   "glcm_ASM_std": asm_s,
    }

features = extract_radiomics(img_uint8, mask_bin)
print(f"Extracted {len(features)} features.")

In [ ]:
# Display features in a formatted table
df_feat = pd.DataFrame([
    {"Feature": k, "Group": "Morphological" if k in ["area_px","eccentricity","solidity","perimeter","bbox_area"]
               else ("Intensity" if k.startswith("int_") else "GLCM Texture"),
     "Value": round(v, 6)}
    for k, v in features.items()
])

for group, sub in df_feat.groupby("Group", sort=False):
    print(f"\n{group}")
    print("-" * 45)
    for _, row in sub.iterrows():
        print(f"  {row['Feature']:35s}: {row['Value']:.6f}")

In [ ]:
# Visualize the segmented region and its bounding box
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original with mask overlay
overlay = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2RGB).copy()
red_channel = np.zeros_like(overlay)
red_channel[:, :, 0] = 255
mask_bool = mask_bin > 0
overlay[mask_bool] = (0.4 * red_channel[mask_bool] + 0.6 * overlay[mask_bool]).astype(np.uint8)

axes[0].imshow(overlay)
axes[0].set_title("Calcium mask overlay")
axes[0].axis("off")

# Morphological features bar chart
morph_keys = ["area_px", "eccentricity", "solidity", "perimeter", "bbox_area"]
morph_vals = [features[k] for k in morph_keys]
axes[1].barh(morph_keys, morph_vals, color="steelblue")
axes[1].set_title("Morphological features")
axes[1].set_xlabel("Value")

# GLCM feature comparison (means)
glcm_props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
glcm_means = [features[f"glcm_{p}_mean"] for p in glcm_props]
axes[2].bar(glcm_props, glcm_means, color="darkorange")
axes[2].set_title("GLCM texture features (means)")
axes[2].set_ylabel("Mean value")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 6. Logistic Regression Classifier

The 23 radiomics features are passed to a **logistic regression** model trained to distinguish
between two Gartner classes:

- **Type I-II** (label 0): dense, well-defined calcifications — higher attenuation values,
  more homogeneous texture (high GLCM energy, low contrast)
- **Type III** (label 1): translucent, ill-defined calcifications — lower attenuation,
  more heterogeneous texture (low energy, high contrast)

### Why logistic regression?
The training dataset for the calcium subtype classification is smaller than the main tendinopathy
dataset. With only 23 features and a limited number of labeled calcification cases, a simple
linear model regularized with L2 penalty is less likely to overfit than a deep network or
even a gradient boosting model. Logistic regression also provides calibrated probabilities
and interpretable coefficients.

The model is serialized with `joblib.dump()` into `clf_lr.joblib`.

In [ ]:
FEATURE_COLS = [
    "area_px", "eccentricity", "solidity", "perimeter", "bbox_area",
    "int_mean", "int_std", "int_min", "int_max", "int_p10", "int_p90",
    "glcm_contrast_mean", "glcm_contrast_std",
    "glcm_dissimilarity_mean", "glcm_dissimilarity_std",
    "glcm_homogeneity_mean", "glcm_homogeneity_std",
    "glcm_energy_mean", "glcm_energy_std",
    "glcm_correlation_mean", "glcm_correlation_std",
    "glcm_ASM_mean", "glcm_ASM_std",
]

try:
    import joblib
    JOBLIB_AVAILABLE = True
except ImportError:
    JOBLIB_AVAILABLE = False

lr_model = None
if JOBLIB_AVAILABLE and CALCIUM_LR_PATH and os.path.exists(CALCIUM_LR_PATH):
    lr_model = joblib.load(CALCIUM_LR_PATH)
    print("Logistic regression classifier loaded.")


def classify_calcium(features_dict, model, feature_cols):
    df = pd.DataFrame([features_dict])
    df = df.reindex(columns=feature_cols, fill_value=0)
    if df.shape[1] != 23:
        raise ValueError(f"Expected 23 features, got {df.shape[1]}")
    prob = float(model.predict_proba(df)[0, 1])
    label_str = "Type III" if prob >= 0.5 else "Type I-II"
    return {"prediction": label_str, "probability": prob}


if lr_model is not None:
    result = classify_calcium(features, lr_model, FEATURE_COLS)
    print(f"Calcium type prediction : {result['prediction']}")
    print(f"Type III probability    : {result['probability']:.4f}")
else:
    print("Skipping classification (no model weights).")
    print("Expected output: {'prediction': 'Type I-II' or 'Type III', 'probability': float}")

## 7. The Full Calcium Pipeline

The `CalciumPipeline` class in `Backend/Calcium_Pipeline.py` orchestrates all four stages.
It is instantiated once at application startup and reused across requests.

```
Input: grayscale crop (from YOLO bounding box region)
          |
          v
  CalciumUNet.predict_mask_from_array()
          |
          v
  Check mask area (< 50 px -> skip, return no prediction)
          |
          v
  CalciumRadiomics.extract_features()  <- 23 hand-crafted features
          |
          v
  CalciumClassifier.predict()          <- logistic regression
          |
          v
  Output: {"prediction": "Type I-II" or "Type III", "mask": np.ndarray}
```

The pipeline returns the prediction string and the binary mask. The calling route in
`app.py` uses the mask to draw a semi-transparent red overlay on the original image,
highlighting the detected calcium deposit to the radiologist.

## 8. Training Context (Conceptual)

### YOLO fine-tuning
Both YOLO models were fine-tuned from YOLOv8n (nano) or YOLOv8s (small) pretrained weights.
The training dataset consisted of annotated shoulder X-rays with bounding boxes drawn around
the supraspinatus and infraspinatus insertion areas. Images were split by projection type
(RI / RE) to train the two separate models.

### Calcium U-Net training
The calcium segmentation U-Net was trained on images where radiologists manually outlined
the calcium deposits pixel by pixel. Due to the small size and irregular shape of deposits,
Dice loss (rather than binary cross-entropy) was used to handle class imbalance between
calcium and non-calcium pixels.

### Radiomics + LR training
1. A dataset of confirmed calcification cases with Gartner type labels was collected.
2. For each case, the calcium mask (from U-Net inference or manual annotation) was used to
   extract the 23 radiomics features.
3. The logistic regression model was trained with L2 regularization and evaluated using
   leave-one-out cross-validation given the small dataset size.

## 9. Summary

The calcium detection pipeline chains four specialized components:

| Stage | Model | Task | Output |
|-------|-------|------|--------|
| 1 | YOLO RE + YOLO RI | Detect supr/infr landmarks | Bounding box + class |
| 2 | Calcium U-Net | Segment calcium pixels | Binary mask (512x512) |
| 3 | Radiomics extractor | Quantify lesion properties | 23-feature vector |
| 4 | Logistic regression | Classify Gartner type | Type I-II or Type III |

The combination of a learned segmentation step (U-Net) with classical hand-crafted features
(radiomics) and a simple linear classifier is a deliberate design choice: it provides
interpretability at the classification stage while still benefiting from the representational
power of deep learning for the difficult pixel-level segmentation task.

The next notebook (05) covers how all of these components are integrated into a production
web API and deployed as a Docker container on HuggingFace Spaces.